# $K$-Means Clustering

## Reading in the Data

In [ ]:
import numpy as np
import pandas as pd

data_dir = "https://dlsun.github.io/stats112/data/"
df_penguins = pd.read_csv(data_dir + "penguins.csv")
df_penguins

Let's focus on just two of the variables, the bill depth and flipper length, so that we can easily visualize the data. Based on the scatterplot below, how many clusters are there in this data set? Can you devise an algorithm that would automatically identify those clusters?

In [ ]:
X_train = df_penguins[["bill_depth_mm", "flipper_length_mm"]].dropna()
X_train.plot.scatter(x="bill_depth_mm", y="flipper_length_mm",
                     c="black", marker="x", alpha=.5)

These two variables are on very different scales, so we will need to scale them first.

In [ ]:
X_train_scaled = (X_train - X_train.mean()) / X_train.std()

## Implementing K-Means from Scratch

$K$-means is an algorithm for finding clusters in data. The idea behind $k$-means is simple: each cluster has a "center" point called the **centroid**, and each observation is assigned to the cluster of its nearest centroid. The challenge is finding the right centroids. The $k$-means algorithm starts with a random guess for the centroids and iteratively improves them.

The steps are as follows:

1. Initialize $k$ centroids at random.
2. Assign each point to the cluster of its nearest centroid.
3. (After reassignment, each centroid may no longer be at the center of its cluster.) Recompute each centroid based on the points assigned to its cluster.
4. Repeat steps 2 and 3 until no points change clusters.

First, we will implement the $k$-means algorithm from scratch. First, let's sample 3 observations at random from the penguins data to serve as the initial centroids.

In [ ]:
# Initialize 3 centroids at random from the data.
centroids = X_train_scaled.sample(3)

# Call the three clusters "red", "blue", "yellow" for convenience.
centroids.index = ["r", "b", "y"]

# Plot these centroids.
ax = X_train_scaled.plot.scatter(x="bill_depth_mm", y="flipper_length_mm",
                                 c="black", marker="x", alpha=.5)
centroids.plot.scatter(x="bill_depth_mm", y="flipper_length_mm",
                       c=centroids.index, ax=ax)

centroids

Now we assign each point to the cluster of its nearest centroid.

In [ ]:
obs = X_train_scaled.loc[0]
np.sqrt(((obs - centroids) ** 2).sum(axis=1)).idxmin()

In [ ]:
# Finds the nearest centroid to a given observation.
def get_nearest_centroid(obs):
    dists = np.sqrt(((obs - centroids) ** 2).sum(axis=1))
    return dists.idxmin()

get_nearest_centroid(X_train_scaled.loc[0])

In [ ]:
# Apply the function to the entire data set.
clusters = X_train_scaled.apply(get_nearest_centroid, axis=1)

# Plot the cluster assignments.
ax = X_train_scaled.plot.scatter(x="bill_depth_mm", y="flipper_length_mm",
                                 c=clusters, marker="x", alpha=.5)
centroids.plot.scatter(x="bill_depth_mm", y="flipper_length_mm",
                       c=centroids.index, ax=ax)

Notice that some of the centroids are no longer at the center of their clusters. We can fix that by redefining the centroid to be the mean of the points in its cluster.

In [ ]:
# Calculate the mean length and width for each cluster.
centroids = X_train_scaled.groupby(clusters).mean()

# Let's plot the new centroids.
ax = X_train_scaled.plot.scatter(x="bill_depth_mm", y="flipper_length_mm",
                                 c=clusters, marker="x", alpha=.5)
centroids.plot.scatter(x="bill_depth_mm", y="flipper_length_mm",
                       c=centroids.index, ax=ax)

centroids

Now, there may be some points that are no longer assigned to their closest centroid, so we have to go back and re-assign clusters. But that may cause the centroids to no longer be at the center of their cluster, so we have to recalculate the centroids. And so on. This process continues until the cluster assignments stop changing.

In [ ]:
# Assign points to their nearest centroid.
clusters = X_train_scaled.apply(get_nearest_centroid, axis=1)

# Recalculate the centroids based on the clusters.
centroids = X_train_scaled.groupby(clusters).mean()

# Plot the current cluster assignments and the centroids.
ax = X_train_scaled.plot.scatter(x="bill_depth_mm", y="flipper_length_mm",
                                 c=clusters, marker="x", alpha=.5)
centroids.plot.scatter(x="bill_depth_mm", y="flipper_length_mm",
                       c=centroids.index, ax=ax)

We can run the code in the above cell over and over until the clusters stop changing. This is the final cluster assignment.

# $K$-Means in _scikit-learn_

We rarely need to implement the $k$-means algorithm from scratch because it is available in _scikit-learn_. The API for _scikit-learn_'s $k$-means model is similar to the API for supervised learning models, like $k$-nearest neighbors, except that the `.fit()` method only takes in `X`, not `X` and `y`. This makes sense because in unsupervised learning, there are no labels `y` in the training data.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

model = KMeans(n_clusters=10)
pipeline = make_pipeline(
    StandardScaler(),
    model
)

pipeline.fit(X_train)

In [ ]:
model.cluster_centers_

In [ ]:
# Extract the centroids and the clusters.
centroids = model.cluster_centers_
clusters = model.labels_

clusters

In [ ]:
# Map the cluster numbers to colors.
clusters = pd.Series(clusters).map({
    0: "r",
    1: "b",
    2: "y",
    3: "g",
    4: "c",
    5: "m",
    6: "k",
    7: "orange",
    8: "purple",
    9: "pink"
})

# Plot the data
X_train.plot.scatter(x="bill_depth_mm", y="flipper_length_mm",
                     c=clusters, marker="x", alpha=.5)

We can call `.predict()` to get the cluster assignment for a new observation. For example, consider a penguin with a bill depth of 15 mm and a flipper length of 210 mm. Visually, it's obvious which cluster this penguin should be assigned to. Let's check that this penguin is indeed assigned to that cluster.

In [ ]:
pipeline.predict([[15, 210]])

Note that `.predict()` simply assigns the test observation to the nearest cluster without recalculating the centroids. (If this observation had been in the training data, then assigning it to a cluster would move the centroid, which in turn would change the assignment of the other points to clusters, and so on.)